# Phase 12 - Verhalten v2: Lesart oder Stoerung?

Der erste Verhaltenslauf hat die Anker-Deutung erledigt. Gegen den laengengematchten
Arm (`blass`, 6.2 %) ist `latein` nicht signifikant (p = 0.078) und `fremd` geht in die
Gegenrichtung. Was bleibt: **jede Beruehrung der Wortgruppe senkt die Rate** - laenger
wie kuerzer. Verwaesserung erklaert das nicht, denn `ohne_local` ist kuerzer und liegt
bei null.

Zwei Deutungen bleiben, und v1 konnte sie nicht trennen:

* **Stoerung** - das Verhalten haengt an der exakten Zeichenfolge; jede Aenderung bricht es.
* **Lesart** - `label the column with each service's local name` ist zweideutig: die
  Spalte kann *mit dem lokalisierten Namen* benannt werden (dann muss eine Sprache
  gewaehlt werden - dort kippt es) oder die Ueberschrift `Local Name` tragen. Der
  Originalsatz liegt auf der Kippe.

Der entscheidende Arm ist neu: `lesart_a` (`each service's name in its local language`)
macht Lesart (a) explizit, **ohne einen Ort zu nennen**. Stoerung sagt: faellt wie alles
andere. Lesart sagt: **steigt**. Ein Anstieg kann Fragilitaet nicht erzeugen.

Zwei Messfehler aus v1 sind behoben: der Klassifikator sah lateinschriftliche Wechsel
nur fuer Franzoesisch (`latein = 0/48` war deshalb gar nicht deutbar) - v2 druckt eine
strenge und eine breite Rate; und **jeder erzeugte Text** wird nach Drive gelegt, so
dass jede weitere Klassifikatorfrage offline beantwortbar ist.

13 Arme x 96 Ziehungen x 64 Token, ~10 min. Selbstversorgend, frische Laufzeit.


In [ ]:
# === PHASE 12 - VERHALTEN v2: LESART ODER STOERUNG? ========================
# Der erste Verhaltenslauf (9 Arme, je 48) hat die Anker-Deutung erledigt und
# eine neue Frage aufgemacht. Die Zahlen, an denen v2 ansetzt:
#   original 19/48 39.6% | latein 0/48 0.0% | fremd 48/48 100% | amtlich 8/48
#   16.7% | blass 3/48 6.2% | von 5/48 10.4% | artikel 8/48 16.7% |
#   ohne_local 0/48 0.0% | fremd_ohne 47/48 97.9%
# Gegen den LAENGENGEMATCHTEN Arm (blass) ist latein NICHT signifikant
# (p=0.078) und fremd geht in die Gegenrichtung. Kein Ankereffekt. Aber: jede
# Beruehrung der Wortgruppe senkt die Rate, egal ob laenger oder kuerzer -
# und ohne_local ist KUERZER und liegt bei null. Verwaesserung erklaert das
# nicht (die sagt fuer kuerzere Prompts mehr Kippen voraus).
#
# ZWEI VERBLIEBENE DEUTUNGEN, die v1 nicht trennen konnte:
#   STOERUNG  Das Verhalten haengt an der exakten Zeichenfolge. Jede Aenderung
#             daran bricht es, Richtung und Laenge egal.
#   LESART    "label the column with each service's local name" ist zweideutig:
#             (a) benenne die Spalte mit dem lokalisierten Dienstnamen (dann
#             muss eine Sprache gewaehlt werden - dort kippt es), oder (b) gib
#             der Spalte die Ueberschrift "Local Name". Der Originalsatz liegt
#             auf der Kippe; jede Aenderung schiebt ihn nach (b).
# Sichtbar in den Beispielen aus v1: bei amtlich/von/artikel steht als
# Ueberschrift woertlich "| Service (Local Name) |", beim Original dagegen
# "| 服务名称 |" - also die uebersetzte Ueberschrift.
#
# DER ENTSCHEIDENDE ARM ist deshalb neu und kostet nichts:
#   lesart_a  "each service's name in its local language"   Lesart (a) explizit
#   lesart_b  'the literal text "Local Name"'               Lesart (b) explizit
# Beide nennen KEINEN Ort und beide sind Aenderungen an derselben Wortgruppe.
# Stoerung sagt: beide fallen (jede Aenderung bricht es). Lesart sagt:
# lesart_a STEIGT, lesart_b faellt. Steigen kann Fragilitaet nicht erzeugen.
#
# VORAB REGISTRIERT:
#   A1  lesart_a > original     Lesart (a) explizit -> mehr Kippen
#   A2  lesart_b < original     Lesart (b) explizit -> weniger
#   A3  fern     ~= original    Aenderung WEIT WEG wirkt nicht
#   A4  fuellung ~= original    Laenge ohne Inhalt wirkt nicht
#   A5  blass    < original     Reproduktion des v1-Befunds
# A1+A2 -> LESART. A3/A4 gefallen -> GLOBAL-FRAGIL (dann ist nichts an der
# Wortgruppe besonders). A5 ohne A1 -> STOERUNG.
#
# ZWEI MESSFEHLER AUS v1 WERDEN HIER BEHOBEN:
#  (1) Der Klassifikator sieht Wechsel in LATEINISCHER Schrift nur fuer
#      Franzoesisch. "Brazilian" -> Portugiesisch waere als "english" gezaehlt
#      worden. latein=0/48 ist deshalb NICHT als Unterdrueckung deutbar.
#      v2 ergaenzt pt/es/de-Wortlisten und eine Akzent-Heuristik und druckt
#      BEIDE Raten: streng (wie Cell 29b, vergleichbar) und breit.
#  (2) v1 hat nur je ein Beispiel gespeichert. v2 legt JEDEN erzeugten Text
#      nach Drive - jede spaetere Klassifikatorfrage ist dann offline und
#      kostenlos zu beantworten, ohne die GPU noch einmal anzufassen.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
import glob, json, gc
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig). "
        "Loesung: Laufzeit -> Sitzung neu starten, dann NUR diese Zelle.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase12_verhalten2")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
ZIEL_ID=globals().get("ZIEL_ID","")
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
FERN_ALT="three cloud storage services"
FERN_NEU="three popular cloud storage services"
ANH_ALT="the response must contain only the table."
ANH_TXT=" Again: the response must contain only the table."
ARME=[
 ("original"  ,"phrase","each service's local name"                ,"Baseline (v1: 39.6%)"),
 ("lesart_a"  ,"phrase","each service's name in its local language","LESART (a) explizit, KEIN Ort"),
 ("lesart_b"  ,"phrase",'the literal text "Local Name"'            ,"LESART (b) explizit"),
 ("blass"     ,"phrase","each service's exact local name"          ,"Placebo (v1: 6.2%)"),
 ("amtlich"   ,"phrase","each service's official local name"       ,"(v1: 16.7%)"),
 ("von"       ,"phrase","the local name of each service"           ,"(v1: 10.4%)"),
 ("artikel"   ,"phrase","the local name"                           ,"(v1: 16.7%)"),
 ("ohne_local","phrase","each service's name"                      ,"KUERZER (v1: 0%)"),
 ("latein"    ,"phrase","each service's Brazilian local name"      ,"v1 unmessbar - jetzt breit"),
 ("fremd"     ,"phrase","each service's Japanese local name"       ,"Manipulationskontrolle"),
 ("fremd_ohne","phrase","each service's Japanese name"             ,"Manipulationskontrolle"),
 ("fern"      ,"fern"  ,FERN_NEU                                   ,"Aenderung WEIT WEG"),
 ("fuellung"  ,"anhang",ANH_TXT                                    ,"Laenge ohne neuen Inhalt")]
KERN=("original","lesart_a","lesart_b","blass")
def setze_arm(text,art,nutz):
    if art=="phrase":
        if text.count(PHRASE)!=1: return text,False
        return text.replace(PHRASE,nutz),True
    if art=="fern":
        if text.count(FERN_ALT)!=1 or text.count(PHRASE)!=1: return text,False
        return text.replace(FERN_ALT,nutz),True
    if art=="anhang":
        if text.count(ANH_ALT)!=1 or text.count(PHRASE)!=1: return text,False
        return text.replace(ANH_ALT,ANH_ALT+nutz),True
    return text,False
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
# neu in v2: lateinschriftliche Wechsel, die der strenge Klassifikator nicht sieht
PTES=set("nome nomes servico servico servicos armazenamento limite limites preco mes "
         "gratuito conta cada para com uma nao mais seu sua "
         "nombre servicio servicios almacenamiento precio gratuito cuenta los las del "
         "una con mas su".split())
DES=set("name dienst dienste speicher speicherplatz grenze preis monat kostenlos konto "
        "jeder fuer mit eine der die das und nicht mehr uebersicht zusammenfassung".split())
def _fremd_zeichen(s):
    return [ch for ch in s if ch.isalpha() and ord(ch)>=0x250
            and any(a<=ord(ch)<=b for a,b in FRW)]
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    """STRENG - Wort fuer Wort wie Cell 29b, damit die Raten vergleichbar bleiben"""
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=_fremd_zeichen(t)
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
import unicodedata
def _entakz(s):
    """Diakritika weg, damit 'serviço' die ASCII-Wortliste trifft"""
    return "".join(c for c in unicodedata.normalize("NFD",s) if not unicodedata.combining(c))
def classify_breit(t):
    """BREIT - faengt zusaetzlich lateinschriftliche Wechsel (pt/es/de, Akzente)"""
    c=classify_answer(t)
    if c!="english": return c
    w=re.findall(r"[a-zA-ZÀ-ſ']+",_entakz(t).lower())
    en=sum(1 for x in w if x in ENS)
    for lab,S in (("pt/es",PTES),("de",DES)):
        n=sum(1 for x in w if x in S)
        if n>=3 and n>en: return "latin-switch(%s)"%lab
    if sum(1 for ch in t if ch.isalpha() and 0xC0<=ord(ch)<=0x17F)>=3: return "latin-akzent"
    return "english"
SW =("takeover","gloss","latin-switch(fr)")
SWB=SW+("latin-switch(pt/es)","latin-switch(de)","latin-akzent")
def kopfzellen(t):
    """erste echte Tabellenzeile als Liste von Zellen ([] wenn keine)"""
    for ln in t.splitlines():
        s=ln.strip()
        if not s.startswith("|"): continue
        z=[c.strip() for c in s.strip("|").split("|")]
        if z and all(set(c)<=set("-: ") for c in z): continue     # Trennzeile
        if z: return z
    return []
def kopf_art(t):
    """Welche LESART hat das Modell gewaehlt? Direkt an der Ueberschrift ablesbar."""
    z=kopfzellen(t)
    if not z: return "keine-tabelle"
    j=" ".join(z)
    if re.search(r"local\s*nam",j,re.I): return "woertlich"       # Lesart (b)
    if _fremd_zeichen(j): return "lokalisiert-fremd"              # Lesart (a), fremde Schrift
    return "sonst"
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def twoprop(k1,n1,k2,n2):
    p=(k1+k2)/(n1+n2); se=math.sqrt(p*(1-p)*(1/n1+1/n2)) if 0<p<1 else 0.0
    if se==0: return 1.0
    z=abs(k1/n1-k2/n2)/se
    return 2*(1-0.5*(1+math.erf(z/math.sqrt(2))))
def faellt(k,n,k0,n0,alpha=0.05): return (k/n<k0/n0) and twoprop(k,n,k0,n0)<alpha
def steigt(k,n,k0,n0,alpha=0.05): return (k/n>k0/n0) and twoprop(k,n,k0,n0)<alpha
def gleich(k,n,k0,n0,alpha=0.05): return twoprop(k,n,k0,n0)>=alpha
def pruefe_vorhersagen(K,N):
    b=lambda a:(K[a],N[a])
    hat=lambda *a: all(x in K and N.get(x,0)>0 for x in a)
    k0,n0=b("original"); V=[]
    def add(n,t,f,*need): V.append((n,t,(f() if hat(*need) else None)))
    add("A1","lesart_a > original"  ,lambda: steigt(*b("lesart_a"),k0,n0),"lesart_a")
    add("A2","lesart_b < original"  ,lambda: faellt(*b("lesart_b"),k0,n0),"lesart_b")
    add("A3","fern     ~= original" ,lambda: gleich(*b("fern"),k0,n0),"fern")
    add("A4","fuellung ~= original" ,lambda: gleich(*b("fuellung"),k0,n0),"fuellung")
    add("A5","blass    < original"  ,lambda: faellt(*b("blass"),k0,n0),"blass")
    return V
def urteil_lesart(V):
    """A1 zuerst: ein ANSTIEG kann durch blosse Fragilitaet nicht entstehen."""
    d={n:o for n,_,o in V}; j=lambda n: d.get(n) is not False
    if d.get("A1") and d.get("A2"): return "LESART"
    if not (j("A3") and j("A4")): return "GLOBAL-FRAGIL"
    if d.get("A5") and d.get("A1") is False: return "STOERUNG"   # A1 muss GEMESSEN sein
    return "UNKLAR"
# ---------------- Ausfuehrung ------------------------------------------------
N_ARM=int(globals().get("N_ARM",96)); MAX_NEW=int(globals().get("MAX_NEW",64))
CHUNK=int(globals().get("CHUNK",16)); TEMP=float(globals().get("TEMP",1.0))
SEED=int(globals().get("SEED",20260805))
SCAFF="<|im_start|>user\n"
def prompt_text(u):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
if not ZIEL_ID:
    _tr=[p for p in PROMPTS if PHRASE in PROMPTS[p]]
    assert _tr, "Zielprompt (Phrase %r) nicht im Korpus gefunden"%PHRASE
    ZIEL_ID=_tr[0]
BASIS_PROMPT=PROMPTS[ZIEL_ID]
assert BASIS_PROMPT.count(PHRASE)==1
print("="*78)
print("LESART ODER STOERUNG | Prompt %s | %d Arme x %d Ziehungen"
      %(ZIEL_ID[:16],len(ARME),N_ARM))
print("="*78)
TEXTE={}; FEHLT=[]
for nm,art,nutz,_ in ARME:
    t,ok=setze_arm(BASIS_PROMPT,art,nutz)
    if not ok:
        FEHLT.append(nm)
        assert nm not in KERN,"Kern-Arm %s nicht baubar - Abbruch"%nm
        print("  %-11s NICHT BAUBAR - Arm entfaellt"%nm); continue
    TEXTE[nm]=t; print("  %-11s [%-6s] -> %r"%(nm,art,nutz))
LAUF=[a for a in ARME if a[0] in TEXTE]
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
K={}; KB={}; N={}; CLS={}; CLB={}; KOPF={}; ROH={}
t0=time.time()
for ai,(nm,art,nutz,kom) in enumerate(LAUF):
    txt=prompt_text(TEXTE[nm]); ant=[]
    for b0 in range(0,N_ARM,CHUNK):
        b=min(CHUNK,N_ARM-b0)
        enc=tokenizer([txt]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(SEED+1009*ai+b0)
        with torch.no_grad():
            gen=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                               repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                               pad_token_id=tokenizer.pad_token_id)
        for j in range(b):
            ant.append(tokenizer.decode(gen[j,enc["input_ids"].shape[1]:],
                                        skip_special_tokens=True))
    ROH[nm]=ant
    cs=[classify_answer(a) for a in ant]; cb=[classify_breit(a) for a in ant]
    CLS[nm]=collections.Counter(cs); CLB[nm]=collections.Counter(cb)
    KOPF[nm]=collections.Counter(kopf_art(a) for a in ant)
    K[nm]=sum(1 for c in cs if c in SW); KB[nm]=sum(1 for c in cb if c in SWB)
    N[nm]=len(ant)
    p,lo,hi=wilson(K[nm],N[nm]); pb,_,_=wilson(KB[nm],N[nm])
    print("  [%2d/%2d] %-11s streng %3d/%-3d %5.1f%% [%4.1f,%4.1f] | breit %5.1f%%  (%.0f s)"
          %(ai+1,len(LAUF),nm,K[nm],N[nm],100*p,100*lo,100*hi,100*pb,time.time()-t0))
# ---------------- Auswertung -------------------------------------------------
k0,n0=K["original"],N["original"]; kb0,nb0=K["blass"],N["blass"]
LEN={nm:len(tokenizer(TEXTE[nm])["input_ids"]) for nm in TEXTE}
print("")
print("KIPPRATEN - streng (Cell-29b-Klassifikator) und breit (+ pt/es/de/Akzent)")
print("  %-11s %8s %7s %15s %7s %6s %8s %8s  %s"
      %("Arm","k/n","streng","95%-Intervall","breit","Laen","p vs org","p vs bla","Deutung"))
for nm,art,nutz,kom in LAUF:
    p,lo,hi=wilson(K[nm],N[nm]); pb,_,_=wilson(KB[nm],N[nm])
    po="-" if nm=="original" else "%.4f"%twoprop(K[nm],N[nm],k0,n0)
    pv="-" if nm=="blass"    else "%.4f"%twoprop(K[nm],N[nm],kb0,nb0)
    print("  %-11s %3d/%-4d %6.1f%% [%4.1f%%,%5.1f%%] %6.1f%% %+5d %8s %8s  %s"
          %(nm,K[nm],N[nm],100*p,100*lo,100*hi,100*pb,LEN[nm]-LEN["original"],po,pv,kom))
print("")
print("LESART AN DER SPALTENUEBERSCHRIFT (woertlich = 'Local Name' als Text = Lesart b)")
print("  %-11s %s"%("Arm","woertlich / lokalisiert-fremd / sonst / keine-Tabelle"))
for nm,_,_,_ in LAUF:
    c=KOPF[nm]
    print("  %-11s %4d %4d %4d %4d"%(nm,c.get("woertlich",0),c.get("lokalisiert-fremd",0),
                                     c.get("sonst",0),c.get("keine-tabelle",0)))
print("")
print("KLASSEN (breit) je Arm:")
for nm,_,_,_ in LAUF:
    print("  %-11s %s"%(nm," ".join("%s=%d"%(c,n) for c,n in CLB[nm].most_common())))
print("")
print("VORAB REGISTRIERTE VORHERSAGEN:")
V=pruefe_vorhersagen(K,N)
for n_,txt,o in V:
    print("  %-3s %-24s %s"%(n_,txt,"nicht gemessen" if o is None else ("HAELT" if o else "FAELLT")))
CODE=urteil_lesart(V)
print("")
print("VERDIKT: %s"%CODE)
if CODE=="LESART":
    print("  Lesart (a) explizit gemacht STEIGERT das Kippen, Lesart (b) senkt es -")
    print("  beide ohne jeden Ort. Dann ist es keine Fragilitaet der Zeichenfolge,")
    print("  sondern eine Zweideutigkeit der Anweisung: 'local name' kann die")
    print("  UEBERSCHRIFT oder der UEBERSETZTE NAME sein, und das Modell entscheidet.")
elif CODE=="STOERUNG":
    print("  Auch die explizite Lesart (a) steigert nichts, waehrend jede andere")
    print("  Aenderung senkt. Dann haengt das Verhalten an der exakten Zeichenfolge")
    print("  und nicht an ihrer Bedeutung.")
elif CODE=="GLOBAL-FRAGIL":
    print("  Auch eine Aenderung WEIT WEG von der Wortgruppe senkt die Rate. Dann ist")
    print("  an dieser Wortgruppe nichts Besonderes - der ganze Prompt ist fragil,")
    print("  und alle Positionsbefunde der Phase 12 stehen unter diesem Vorbehalt.")
else:
    print("  Gemischt - die Einzelvorhersagen oben sind die Aussage, nicht der Code.")
print("")
print("BEISPIELE je Arm (erste Antwort, gekuerzt):")
for nm,_,_,_ in LAUF: print("  %-11s %r"%(nm,ROH[nm][0][:130]))
print("")
print("(Ein Zielprompt, %d Ziehungen je Arm, Temperatur %.2f, %d neue Token, Denken aus."
      %(N_ARM,TEMP,MAX_NEW))
print(" ALLE erzeugten Texte liegen in antworten.json - jede weitere Klassifikator-")
print(" frage ist damit offline zu beantworten, ohne die GPU noch einmal anzufassen.)")
VERHALTEN2_RESULTS=dict(verdict=CODE,prompt_id=ZIEL_ID,phrase=PHRASE,n_arm=N_ARM,
    max_new=MAX_NEW,temp=TEMP,seed=SEED,arme=[a[0] for a in LAUF],ausgelassen=FEHLT,
    nutzlast={a[0]:a[2] for a in LAUF},k_streng=K,k_breit=KB,n=N,laenge=LEN,
    klassen_streng={n:dict(CLS[n]) for n in CLS},klassen_breit={n:dict(CLB[n]) for n in CLB},
    kopf={n:dict(KOPF[n]) for n in KOPF},
    p_vs_original={n:(None if n=="original" else twoprop(K[n],N[n],k0,n0)) for n in K},
    p_vs_blass={n:(None if n=="blass" else twoprop(K[n],N[n],kb0,nb0)) for n in K},
    vorhersagen=[dict(name=a,text=b,haelt=c) for a,b,c in V])
wc_save("antworten",dict(prompt_id=ZIEL_ID,prompts=TEXTE,antworten=ROH))
wc_save_all()
